🟡 Intermédiaire | ⏱ 90 min | 🔑 yield, itertools, import, pip, datetime, zoneinfo

# 04 - Générateurs, modules et dates

## 1. Itérateurs et générateurs

Un **itérable** fournit un itérateur (`iter()`), qu'on consomme avec `next()` jusqu'à `StopIteration`. Un **générateur** est une fonction avec `yield` : elle produit les valeurs **à la demande** (lazy) sans tout stocker en mémoire.

In [ ]:
# Protocole d'itération : iter() puis next() jusqu'à StopIteration
it = iter([10, 20])
print(next(it), next(it))
try:
    next(it)
except StopIteration:
    print("épuisé")

In [ ]:
# yield suspend la fonction et reprend au prochain next()
def compte_a_rebours(n):
    while n > 0:
        yield n
        n -= 1

g = compte_a_rebours(3)
print(type(g))
print(list(g))

In [ ]:
# Un générateur ne se rejoue pas : une fois consommé, il est vide
print(list(g))

In [ ]:
# Lazy vs eager : la liste alloue tout, le générateur calcule au fil de l'eau
import sys
liste = [n * n for n in range(1_000_000)]
gen = (n * n for n in range(1_000_000))
print(sys.getsizeof(liste), sys.getsizeof(gen))
print(sum(gen))

In [ ]:
# Pipeline de générateurs : chaque étape ne traite qu'une ligne à la fois
def lire(lignes):
    for l in lignes:
        yield l.strip()

def filtrer(lignes, prefixe):
    return (l for l in lignes if l.startswith(prefixe))

brut = ["  ERROR disque plein", "INFO ok", "ERROR réseau  "]
print(list(filtrer(lire(brut), "ERROR")))

In [ ]:
# itertools : itérateur infini tranché avec islice
import itertools as it
print(list(it.islice(it.count(10, 5), 4)))

In [ ]:
# chain : enchaîner plusieurs itérables
print(list(it.chain([1, 2], (3, 4))))

In [ ]:
# product et combinations : combinatoire
print(list(it.product("AB", [1, 2])))
print(list(it.combinations([1, 2, 3], 2)))

In [ ]:
# groupby : regrouper les éléments consécutifs identiques
print([(k, list(g)) for k, g in it.groupby("aabbbc")])

In [ ]:
# accumulate : sommes cumulées
print(list(it.accumulate([1, 2, 3, 4])))

## 2. Modules et packages

Un **module** est un fichier `.py` ; un **package** est un dossier avec `__init__.py`. `import` exécute le module une seule fois et le met en cache dans `sys.modules`.

```
mon_projet/
├── main.py
└── outils/
    ├── __init__.py
    ├── maths.py       → from outils.maths import moyenne
    └── texte.py       → from outils import texte
```

Formes d'import, de la plus explicite à la moins recommandée :
```python
import math                 # math.sqrt(2)
import numpy as np          # alias
from math import sqrt, pi   # sélectif
from math import *          # à éviter : pollue l'espace de noms
```
Ordre PEP 8 : bibliothèque standard, puis packages tiers, puis modules locaux.

In [ ]:
# Créer un module dans un dossier temporaire (un module = un fichier .py)
import sys, tempfile
from pathlib import Path

dossier = Path(tempfile.mkdtemp())
(dossier / "outils.py").write_text('''
"""Petit module utilitaire."""
VERSION = "1.0"

def moyenne(valeurs):
    return sum(valeurs) / len(valeurs)

if __name__ == "__main__":        # vrai seulement si lancé avec `python outils.py`
    print("test :", moyenne([1, 2, 3]))
''')
print(dossier / "outils.py")

In [ ]:
# Rendre le dossier visible pour import (équivalent de PYTHONPATH), puis importer
sys.path.insert(0, str(dossier))

import outils
print(outils.__name__, outils.VERSION, outils.moyenne([10, 20]))

In [ ]:
# dir() : ce que le module expose
print([n for n in dir(outils) if not n.startswith("_")])

### pip et environnements virtuels

```bash
python -m venv .venv                  # créer l'environnement
source .venv/bin/activate             # Windows : .venv\Scripts\activate
pip install requests pandas==2.2.0    # installer
pip freeze > requirements.txt         # figer les versions
pip install -r requirements.txt       # réinstaller ailleurs
pip list / pip show requests / pip uninstall requests
```

Modules standard à connaître : `os`, `sys`, `pathlib`, `json`, `csv`, `re`, `math`, `random`, `datetime`, `collections`, `itertools`, `functools`, `logging`.

In [ ]:
# json : sérialiser / désérialiser
import json
data = {"nom": "Alice", "scores": [12, 15]}
texte = json.dumps(data, ensure_ascii=False, indent=2)
print(texte)
print(json.loads(texte)["scores"])

In [ ]:
# random et math
import random, math
random.seed(42)
print(random.randint(1, 6), random.choice(["a", "b"]), math.pi)

In [ ]:
# pathlib : manipuler des chemins
from pathlib import Path
print(Path.cwd().name, Path("data/raw/ventes.csv").suffix)

## 3. Dates et heures (`datetime`)

Classes : `date`, `time`, `datetime`, `timedelta`, `timezone`. Format texte : `strftime()` (date → str) et `strptime()` (str → date). Standard d'échange : **ISO 8601** (`2026-10-12T14:30:00+02:00`).

In [ ]:
# date et datetime : aujourd'hui, maintenant, une date fixée
from datetime import date, datetime, timedelta

aujourd_hui = date.today()
maintenant = datetime.now()
noel = datetime(2026, 12, 25, 18, 0)
print(aujourd_hui, maintenant.year, noel.weekday())   # 0 = lundi

In [ ]:
# La différence entre deux datetime est un timedelta
print(noel - maintenant)
print((noel - maintenant).days, (noel - maintenant).total_seconds())

In [ ]:
# Ajouter une durée avec timedelta
print(noel + timedelta(days=7, hours=6))

In [ ]:
# Formatage : strftime() et isoformat()
# Codes utiles : %Y année, %m mois, %d jour, %H heure, %M minute, %S seconde, %A jour de semaine, %B mois en toutes lettres
d = datetime(2026, 10, 12, 14, 30)
print(d.strftime("%d/%m/%Y %H:%M"))
print(d.strftime("%A %d %B"))
print(d.isoformat())

In [ ]:
# Parsing : strptime() avec un format, fromisoformat() pour l'ISO 8601
print(datetime.strptime("12/10/2026 14:30", "%d/%m/%Y %H:%M"))
print(datetime.fromisoformat("2026-10-12T14:30:00+02:00"))
print(date.fromisoformat("2026-10-12"))

In [ ]:
# Naive (sans fuseau) vs aware (avec fuseau). Règle : stocker en UTC, afficher en local.
from datetime import timezone
from zoneinfo import ZoneInfo

naive = datetime(2026, 7, 1, 12, 0)
utc = datetime(2026, 7, 1, 12, 0, tzinfo=timezone.utc)
print(naive.tzinfo, utc.tzinfo)

In [ ]:
# Convertir vers un fuseau local avec astimezone()
paris = utc.astimezone(ZoneInfo("Europe/Paris"))
tokyo = utc.astimezone(ZoneInfo("Asia/Tokyo"))
print(utc.isoformat())
print(paris.isoformat())
print(tokyo.hour)

In [ ]:
# On ne peut pas comparer naive et aware
try:
    naive < utc
except TypeError as e:
    print("TypeError :", e)

In [ ]:
# Timestamp Unix : secondes depuis 1970-01-01 UTC
ts = utc.timestamp()
print(ts)
print(datetime.fromtimestamp(ts, tz=timezone.utc))

In [ ]:
# L'heure courante en UTC : préférer now(timezone.utc) à utcnow() (déprécié)
print(datetime.now(timezone.utc))

## ⚠️ Pièges

- Un générateur épuisé est vide : le recréer ou le convertir en liste si on doit repasser dessus.
- Imports circulaires (`a` importe `b` qui importe `a`) : regrouper, ou importer dans la fonction.
- Nommer un fichier `random.py` ou `json.py` masque le module standard du même nom.
- Comparer une date naive et une date aware lève `TypeError`.
- L'heure d'été : `+ timedelta(hours=24)` sur une date Paris ne donne pas forcément la même heure locale. Calculer en UTC.
- `strptime` est strict : le format doit correspondre exactement à la chaîne.